In [0]:
%sql

USE CATALOG `retail-dwh-project`;

CREATE SCHEMA IF NOT EXISTS gold;

USE SCHEMA gold;


In [0]:

-- =========================
-- DIM CUSTOMER
-- =========================

CREATE TABLE IF NOT EXISTS gold.dim_customer (
    CustomerSK   BIGINT,
    CustomerID   INT,
    CustomerName STRING,
    Email        STRING,
    City         STRING,
    Address      STRING,
    LastUpdated  DATE,
    StartDate    DATE,
    EndDate      DATE,
    IsActive     INT
);


In [0]:

-- =========================
-- DIM PRODUCT
-- =========================

CREATE TABLE IF NOT EXISTS gold.dim_product (
    ProductSK     BIGINT,
    ProductID     INT,
    ProductName   STRING,
    Category      STRING,
    UnitPrice     DECIMAL(10,2),
    EffectiveDate DATE
);


In [0]:

-- =========================
-- DIM STORE
-- =========================

CREATE TABLE IF NOT EXISTS gold.dim_store (
    StoreSK   BIGINT,
    StoreID   INT,
    StoreName STRING,
    Region    STRING
);


In [0]:

-- =========================
-- FACT SALES
-- =========================

CREATE TABLE IF NOT EXISTS gold.fact_sales (
    SalesSK       BIGINT,
    TransactionID INT,
    CustomerSK    BIGINT,
    ProductSK     BIGINT,
    StoreSK       BIGINT,
    Quantity      INT,
    Amount        DECIMAL(10,2),
    TxnDate       DATE
);


In [0]:

-- =========================
-- LOAD DIM PRODUCT
-- =========================

INSERT OVERWRITE gold.dim_product
SELECT
    ROW_NUMBER() OVER (ORDER BY ProductID) AS ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    CURRENT_DATE() AS EffectiveDate
FROM clean.products_clean;


In [0]:

-- =========================
-- LOAD DIM STORE
-- =========================

INSERT OVERWRITE gold.dim_store
SELECT
    ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreSK,
    StoreID,
    StoreName,
    Region
FROM clean.stores_clean;


In [0]:

-- =========================
-- INITIAL LOAD DIM CUSTOMER
-- =========================

INSERT INTO gold.dim_customer
SELECT
    (SELECT COALESCE(MAX(CustomerSK),0)
     FROM gold.dim_customer)
     + ROW_NUMBER() OVER (ORDER BY CustomerID) AS CustomerSK,

    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated,

    CURRENT_DATE()     AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1                  AS IsActive

FROM clean.customers_clean

WHERE CustomerID NOT IN (
    SELECT CustomerID
    FROM gold.dim_customer
);


In [0]:

-- =========================
-- SCD TYPE 2 UPDATE
-- =========================

UPDATE gold.dim_customer
SET
    IsActive = 0,
    EndDate  = CURRENT_DATE()

WHERE IsActive = 1
AND EXISTS (

    SELECT 1
    FROM clean.customers_clean n

   WHERE n.CustomerID   = dim_customer.CustomerID
      AND n.CustomerName = dim_customer.CustomerName
      AND (n.City != dim_customer.City OR n.Address != dim_customer.Address)
);


In [0]:

-- =========================
-- INSERT NEW CUSTOMER VERSION
-- =========================

INSERT INTO gold.dim_customer
SELECT
    (SELECT COALESCE(MAX(CustomerSK),0)
     FROM gold.dim_customer)
     + ROW_NUMBER() OVER (ORDER BY n.CustomerID) AS CustomerSK,

    n.CustomerID,
    n.CustomerName,
    n.Email,
    n.City,
    n.Address,
    n.LastUpdated,

    CURRENT_DATE()     AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1                  AS IsActive

FROM clean.customers_clean n

JOIN gold.dim_customer d
ON n.CustomerID = d.CustomerID

WHERE d.IsActive = 0
AND d.EndDate = CURRENT_DATE();


In [0]:

-- =========================
-- LOAD FACT SALES
-- =========================

INSERT INTO gold.fact_sales
SELECT
    (SELECT COALESCE(MAX(SalesSK),0)
     FROM gold.fact_sales)
     + ROW_NUMBER() OVER (ORDER BY s.TransactionID) AS SalesSK,

    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,

    ROUND(s.Quantity * p.UnitPrice, 2) AS Amount,

    s.TxnDate

FROM clean.sales_clean s

INNER JOIN gold.dim_customer c
ON s.CustomerID = c.CustomerID
AND c.IsActive = 1

INNER JOIN gold.dim_product p
ON s.ProductID = p.ProductID

INNER JOIN gold.dim_store st
ON s.StoreID = st.StoreID

WHERE s.TransactionID NOT IN (
    SELECT TransactionID
    FROM gold.fact_sales
);


In [0]:

-- =========================
-- VALIDATE GOLD TABLES
-- =========================

SELECT 'dim_customer' AS table_name, COUNT(*) AS total_rows
FROM gold.dim_customer

UNION ALL

SELECT 'dim_product', COUNT(*)
FROM gold.dim_product

UNION ALL

SELECT 'dim_store', COUNT(*)
FROM gold.dim_store

UNION ALL

SELECT 'fact_sales', COUNT(*)
FROM gold.fact_sales;

In [0]:
drop table if exists gold.dim_customer;
